# Lingustics of a projection

In [1]:
# --- spaCy (Morphologizer) ---

import spacy

future_tense = []

nlp = spacy.load("en_core_web_lg")
doc = nlp("He will go running.")
for t in doc:
    tense = t.morph.get("Tense")
    
    if tense == ['Futu']:
        future_tense.append(tense)

    print("spaCy:", t.text, t.morph.get("Tense"))


spaCy: He []
spaCy: will []
spaCy: go []
spaCy: running ['Pres']
spaCy: . []


In [2]:
import spacy
nlp = spacy.load("en_core_web_lg")
doc = nlp("He will go running.")

for t in doc:
    if t.tag_ == "MD" or (t.pos_ == "AUX" and t.lemma_ == "will"):
        print("Modal aux (future-ish):", t.text, {"pos": t.pos_, "tag": t.tag_, "morph": t.morph, "dep": t.dep_})

# Optional: flag the clause as 'future' if MD precedes a base verb
has_future = any(t.tag_ == "MD" and t.head.tag_ in {"VB", "VBP"} for t in doc)
print("future_like_clause:", has_future)

Modal aux (future-ish): will {'pos': 'AUX', 'tag': 'MD', 'morph': VerbForm=Fin, 'dep': 'aux'}
future_like_clause: True


In [3]:
import os
import sys
import warnings
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from tqdm import tqdm

from sklearn.datasets import make_classification
from imblearn.over_sampling import RandomOverSampler, SMOTE

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))

from data_visualizing import DataVisualizing
from data_processing import DataProcessing
from feature_extraction import SpacyFeatureExtraction

## Load Dataset

In [4]:
df = DataProcessing.load_single_synthetic_data(
    notebook_dir, batch_idx=1, sep=',', return_as='dataframe'
)
df = df.loc[:7, :]
df

Loading: /Users/detraviousjamaribrinkley/Documents/Development/research_labs/uf_ds/predictions/notebook_experiments/../data/prediction_logs/batch_1-prediction/batch_1-from_df.csv


,Base Sentence,Sentence Label,Domain,Model Name,API Name,Batch ID,Template Number
0,JPMorgan Chase forecasts that the net profit a...,1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,1
1,"On August 21, 2024, Bank of America speculates...",1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,2
2,"Citigroup predicts on 2024-08-21, the operatin...",1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,3
3,"According to Goldman Sachs, the research and d...",1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,4
4,"In 21 August 2024, Morgan Stanley envisions th...",1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,5
5,The stock price at Visa should stay same in Q2...,1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,6
6,JPMorgan forecasts that the revenue at Microso...,1,finance,llama-3.3-70b-instruct,NAVI_GATOR,0,1
7,"On August 25, 2024, to September 25, 2025, Cit...",1,finance,llama-3.3-70b-instruct,NAVI_GATOR,0,2


## Extract Tense

+ Morphology $ \rightarrow $ morphemes
    1. spaCy: 
        - Technique: Statistical tagger
        - Output: Feature values organized as key-value pairs (e.g., `Number=Sing|Tense=Past`)
        - Note: NOT a full derivational analysis; it reports surface features, not raw morphemes.
        - Note: It doesn’t rebuild the word from morphemes; it infers probabilistically the most likely label set.
        - Note: Doesn't check the construction of the word, instead utilizes probability to determine label/feature value
        - Note: `token.noun_chunks` is for phrase‑level grouping, not morpheme segmentation.
        - Note: `token._.morph_analysis` (via an extension) gives access to the same feature set, not a morpheme list.

    2. Morphological Analysis: 
        - Technique: Explicit morpheme splitter (prefixes, stems, suffixes, infixes, etc.)
        - Output: decomposed structure + #morphemes + phonetics of each morphome
        - Note: Full derivational analysis

In [5]:
disable_components = []
spe = SpacyFeatureExtraction(df, 'Base Sentence')
pos_df, ner_df = spe.extract_pos_ner_features(disable_components=[], visualize=True)
pos_df



Spacy POS extraction: 0it [00:00, ?it/s]


	####### Sentence (0): JPMorgan Chase forecasts that the net profit at Amazon potentially decrease in Q3 of 2027. #######


Spacy POS extraction: 1it [00:00,  4.81it/s]


	####### Sentence (1): On August 21, 2024, Bank of America speculates the revenue at Microsoft will likely increase. #######



	####### Sentence (2): Citigroup predicts on 2024-08-21, the operating income at Alphabet may rise. #######


...


	####### Sentence (1): On August 21, 2024, Bank of America speculates the revenue at Microsoft will likely increase. #######



	####### Sentence (2): Citigroup predicts on 2024-08-21, the operating income at Alphabet may rise. #######



	####### Sentence (3): According to Goldman Sachs, the research and development expenses at Facebook would fall in 2025. #######


8it [00:00, 273.69it/s]


,Sentence,Term,Lemma,POS Label,Detailed POS Label,Dependency,Shape,Is Alpha,Stop Word,Unique POS Label,...,Definite,Degree,SubCat,Animacy,Acc,Gen,Loc,Ins,Parat,Prep
0,"(JPMorgan, Chase, forecasts, that, the, net, p...",JPMorgan,JPMorgan,PROPN,NNP,compound,XXXxxxx,True,False,PROPN_1,...,,,,,,,,,,
1,"(JPMorgan, Chase, forecasts, that, the, net, p...",Chase,Chase,PROPN,NNP,nsubj,Xxxxx,True,False,PROPN_2,...,,,,,,,,,,
2,"(JPMorgan, Chase, forecasts, that, the, net, p...",forecasts,forecast,VERB,VBZ,ROOT,xxxx,True,False,VERB_1,...,,,,,,,,,,
3,"(JPMorgan, Chase, forecasts, that, the, net, p...",that,that,SCONJ,IN,mark,xxxx,True,True,SCONJ_1,...,,,,,,,,,,
4,"(JPMorgan, Chase, forecasts, that, the, net, p...",the,the,DET,DT,det,xxx,True,True,DET_1,...,[Def],,,,,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,"(On, August, 25, ,, 2024, ,, to, September, 25...",will,will,AUX,MD,aux,xxxx,True,True,AUX_5,...,,,,,,,,,,
152,"(On, August, 25, ,, 2024, ,, to, September, 25...",likely,likely,ADV,RB,advmod,xxxx,True,False,ADV_4,...,,,,,,,,,,
153,"(On, August, 25, ,, 2024, ,, to, September, 25...",increase,increase,VERB,VB,ccomp,xxxx,True,False,VERB_15,...,,,,,,,,,,
154,"(On, August, 25, ,, 2024, ,, to, September, 25...",.,.,PUNCT,.,punct,.,False,False,PUNCT_18,...,,,,,,,,,,


In [6]:
ner_df

,Sentence,Term,NER Label,Unique NER Label,Start Char,End Char
0,"(JPMorgan, Chase, forecasts, that, the, net, p...",JPMorgan Chase,ORG,ORG_1,0,14
1,"(JPMorgan, Chase, forecasts, that, the, net, p...",Amazon,ORG,ORG_2,48,54
2,"(JPMorgan, Chase, forecasts, that, the, net, p...",Q3,GPE,GPE_1,79,81
3,"(JPMorgan, Chase, forecasts, that, the, net, p...",2027,DATE,DATE_1,85,89
4,,,,,,
5,"(On, August, 21, ,, 2024, ,, Bank, of, America...","August 21, 2024",DATE,DATE_2,3,18
6,"(On, August, 21, ,, 2024, ,, Bank, of, America...",Bank of America,ORG,ORG_3,20,35
7,"(On, August, 21, ,, 2024, ,, Bank, of, America...",Microsoft,ORG,ORG_4,62,71
8,,,,,,
9,"(Citigroup, predicts, on, 2024, -, 08, -, 21, ...",Citigroup,ORG,ORG_5,0,9


## Filter by what gives future tense
1. NER: 
    1. VERB
        - Unique NER: 
            | Tag | Form | Example | Grammatical Role | Tense (Future)
            |-----|------|---------|-------------------|------|
            | **VB** | Base form | `sleep`, `run` | Infinitive or base form (e.g., “to sleep”, “I sleep”) | ❌
            | **VBD** | Past tense | `slept`, `ran` | Simple past (e.g., “She slept”) | ❌
            | **VBG** | Present participle / gerund | `sleeping`, `running` | Progressive or gerund (e.g., “I am sleeping”, “Sleeping is fun”) | ❌
            | **VBN** | Past participle | `slept`, `run` | Perfect or passive (e.g., “She has slept”, “The house was run”) | ❌
            | **VBP** | Non‑3rd‑person singular present | `sleep`, `run` | “I sleep”, “They run” | ❌
            | **VBZ** | 3rd‑person singular present | `sleeps`, `runs` | “He sleeps” (the ‑s ending) | ❌
    2. AUX
        - Unique NER: 
            | Tag | Form | Example | Grammatical Role | Tense (Future)
            |-----|------|---------|-------------------|------|
            | **MD** | Modal | will | Modal auxiliary indicating future | ✅
            | **MD** | Modal | shall | Modal auxiliary expressing future (especially in formal English) | ✅
            | **MD** | Modal | should | Modal that can express future obligation or advice | ✅
            | **MD** | Modal | may | Modal that can express future possibility | ✅
            | **MD** | Modal | might | Modal that can express future conditionality | ✅
            | **MD** | Modal | could | Modal that can express future potential | ✅

In [7]:
sentence_pos_df = pos_df.loc[:, ['Sentence', 'POS Label']]
filt_aux = (sentence_pos_df['POS Label'] == 'AUX')
sentence_pos_df[filt_aux]

,Sentence,POS Label
31,"(On, August, 21, ,, 2024, ,, Bank, of, America...",AUX
50,"(Citigroup, predicts, on, 2024, -, 08, -, 21, ...",AUX
66,"(According, to, Goldman, Sachs, ,, the, resear...",AUX
101,"(The, stock, price, at, Visa, should, stay, sa...",AUX
151,"(On, August, 25, ,, 2024, ,, to, September, 25...",AUX


## Filter by what gives future tense
- Morphology: Person

In [8]:
sentence_person_df = pos_df.loc[:, ['Sentence', 'Person']]
filt_person = (sentence_person_df['Person'].astype(str) == "['3']")
sentence_person_df[filt_person]

,Sentence,Person
2,"(JPMorgan, Chase, forecasts, that, the, net, p...",[3]
26,"(On, August, 21, ,, 2024, ,, Bank, of, America...",[3]
37,"(Citigroup, predicts, on, 2024, -, 08, -, 21, ...",[3]
79,"(In, 21, August, 2024, ,, Morgan, Stanley, env...",[3]
88,"(In, 21, August, 2024, ,, Morgan, Stanley, env...",[3]
116,"(JPMorgan, forecasts, that, the, revenue, at, ...",[3]
143,"(On, August, 25, ,, 2024, ,, to, September, 25...",[3]


## Filter by what gives future tense

- Morphology: Tense

In [9]:
sentence_tense_df = pos_df.loc[:, ['Sentence', 'Tense']]
filt_person = (sentence_tense_df['Tense'].astype(str) == "['Pres']")
sentence_tense_df[filt_person]

,Sentence,Tense
2,"(JPMorgan, Chase, forecasts, that, the, net, p...",[Pres]
26,"(On, August, 21, ,, 2024, ,, Bank, of, America...",[Pres]
37,"(Citigroup, predicts, on, 2024, -, 08, -, 21, ...",[Pres]
54,"(According, to, Goldman, Sachs, ,, the, resear...",[Pres]
79,"(In, 21, August, 2024, ,, Morgan, Stanley, env...",[Pres]
88,"(In, 21, August, 2024, ,, Morgan, Stanley, env...",[Pres]
109,"(The, stock, price, at, Visa, should, stay, sa...",[Pres]
116,"(JPMorgan, forecasts, that, the, revenue, at, ...",[Pres]
143,"(On, August, 25, ,, 2024, ,, to, September, 25...",[Pres]


In [10]:
df

,Base Sentence,Sentence Label,Domain,Model Name,API Name,Batch ID,Template Number
0,JPMorgan Chase forecasts that the net profit a...,1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,1
1,"On August 21, 2024, Bank of America speculates...",1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,2
2,"Citigroup predicts on 2024-08-21, the operatin...",1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,3
3,"According to Goldman Sachs, the research and d...",1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,4
4,"In 21 August 2024, Morgan Stanley envisions th...",1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,5
5,The stock price at Visa should stay same in Q2...,1,finance,llama-3.1-70b-instruct,NAVI_GATOR,0,6
6,JPMorgan forecasts that the revenue at Microso...,1,finance,llama-3.3-70b-instruct,NAVI_GATOR,0,1
7,"On August 25, 2024, to September 25, 2025, Cit...",1,finance,llama-3.3-70b-instruct,NAVI_GATOR,0,2


In [11]:
sentence_person_df['Tense'].value_counts()

KeyError: 'Tense'

In [ ]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_lg")
doc = nlp("Rats are various medium-sized, long-tailed rodents.")
displacy.render(doc, style="dep", jupyter=True)


In [ ]:
from IPython.display import HTML, display
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_lg")
doc = nlp("Rats are various medium-sized, long-tailed rodents.")

# Get HTML string and display it
html = displacy.render(doc, style="dep", jupyter=False)
display(HTML(html))

In [ ]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_lg")
doc1 = nlp("This is a sentence.")
doc2 = nlp("This is another sentence.")
html = displacy.render([doc1, doc2], style="dep", page=True)